In [17]:
import h5py
import glob
import os
import numpy as np
import pandas as pd
import dask.dataframe as dd
import scipy as sp
import seaborn as sns
import cooler
import umap
import anndata
import harmonypy as hm
from itertools import cycle, islice

import igraph as ig
import leidenalg as la
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import normalize, OneHotEncoder

import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42

import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib.colors import LogNorm
from matplotlib import cm as cm
import cooltools.lib.plotting

from scipy import ndimage as nd
from scipy.stats import pearsonr
from scipy.signal import find_peaks
from sklearn.decomposition import PCA
from statsmodels.sandbox.stats.multicomp import multipletests as FDR
from concurrent.futures import ProcessPoolExecutor, as_completed

from ALLCools.clustering import significant_pc_test

### openBLAS will not work properly in jupyter notebook
os.environ['OPENBLAS_NUM_THREADS'] = '1'

## Prepare gene score annotation

In [15]:
### load gene score
### speed up
gene_hdf = []
for file in [file for file in os.listdir("../impute/genescore/") if file.endswith('.hdf')]:
    tmp = pd.read_hdf("../impute/genescore/" + file, key = "data")
    gene_hdf.append(tmp)
    
dask_df = dd.from_pandas(pd.concat(gene_hdf, axis=0), npartitions=10)
gene_hdf = dask_df.compute()
print(gene_hdf.shape)

(72986, 36559)


In [32]:
gene_hdf1 = []
for file in [file for file in os.listdir("../../87.FNIH_DHC_IGM_240925/impute/genescore/") if file.endswith('.hdf')]:
    tmp = pd.read_hdf("../../87.FNIH_DHC_IGM_240925/impute/genescore/" + file, key = "data")
    gene_hdf1.append(tmp)
    
dask_df = dd.from_pandas(pd.concat(gene_hdf1, axis=0), npartitions=10)
gene_hdf1 = dask_df.compute()
gene_hdf1 = gene_hdf1[gene_hdf1.index.astype(str).str.startswith(('LV009', 'LV012'))]
print(gene_hdf1.shape)

(14374, 36559)


In [33]:
### add liver cells from previous sequencing run
gene_hdf2 = []
for file in [file for file in os.listdir("../../90.FNIH_DHC_IGM_241023/impute/genescore/") if file.endswith('.hdf')]:
    tmp = pd.read_hdf("../../90.FNIH_DHC_IGM_241023/impute/genescore/" + file, key = "data")
    gene_hdf2.append(tmp)
    
dask_df = dd.from_pandas(pd.concat(gene_hdf2, axis=0), npartitions=10)
gene_hdf2 = dask_df.compute()
print(gene_hdf2.shape)

(9733, 36559)


In [34]:
gene_hdf = pd.concat([gene_hdf, gene_hdf1, gene_hdf2], axis = 0)
print(gene_hdf.shape)

(97093, 36559)


In [30]:
res = 10000
gene_meta_path = '/projects/ps-renlab/y2xie/projects/genome_ref/Paired-Tag/hg38/hg38.gcode.10X.txt'
gene_meta = pd.read_csv(gene_meta_path, names = ['chrom', 'start', 'end', 'gene_id', 'gene_name'], index_col='gene_id', sep='\s+')
gene_meta['bin_len'] = (gene_meta['end'] // res) - (gene_meta['start'] // res) + 1
gene_meta['weight'] = (gene_meta['bin_len']+2) * (gene_meta['bin_len'] + 1) / 2

chrom_size_path = '/projects/ps-renlab/y2xie/projects/genome_ref/hg38.main.chrom.sizes'
chromsize = pd.read_csv(chrom_size_path, sep='\t', header=None, index_col=0)
gene_meta = gene_meta[gene_meta['chrom'].isin(chromsize.index)]
gene_meta.head()

,chrom,start,end,gene_name,bin_len,weight
gene_id,,,,,,
ENSG00000243485,chr1,29554,31109,MIR1302-2HG,2,6.0
ENSG00000237613,chr1,34554,36081,FAM138A,1,3.0
ENSG00000186092,chr1,65419,71585,OR4F5,2,6.0
ENSG00000238009,chr1,89295,133723,AL627309.1,6,28.0
ENSG00000239945,chr1,89551,91105,AL627309.3,2,6.0


In [46]:
stat = []
### first need to sotlink all stat
for file in [file for file in os.listdir("../03.mapping/") if file.endswith('.sc.stat.csv')]:
    fname = file[0:5]
    tmp = pd.read_csv('../03.mapping/' + file, sep = '\t', index_col = 0)
    tmp.index = fname + '_' + tmp.index
    stat.append(tmp)
        
stat = pd.concat(stat, axis=0)
print(stat.shape)

(12592135, 9)


In [73]:
# sparse_matrix = csr_matrix(gene_hdf)
gene3c = anndata.AnnData(sparse_matrix, obs = pd.DataFrame(index=gene_hdf.index), 
                         var = pd.DataFrame(index=gene_hdf.columns))
tmeta = gene3c.obs.merge(stat, left_index = True, right_index = True)
gene3c = gene3c[tmeta.index, :]
gene3c.obs = tmeta
gene3c

# genefilter = ((gene3c.X>0).sum(axis=0)>10) & (gene3c.var.index.isin(gene_meta.index))
# gene3c = gene3c[:, genefilter]

AnnData object with n_obs × n_vars = 97093 × 36559
    obs: 'total', 'mapped', 'unmapped', 'duplicate', 'cis', 'cis_1kb-', 'cis_1kb+', 'cis_10kb+', 'trans'

In [75]:
from scipy.sparse import csr_matrix
import scanpy as sc
gene3c.var.index.name = 'gene_id'
gene3c.var.index = gene_meta.loc[gene3c.var.index, 'gene_name']
gene3c.var_names_make_unique()
gene3c

AnnData object with n_obs × n_vars = 97093 × 36559
    obs: 'total', 'mapped', 'unmapped', 'duplicate', 'cis', 'cis_1kb-', 'cis_1kb+', 'cis_10kb+', 'trans'

In [78]:
gene3c.write_h5ad('integration/FNIH_DHC_genescore.raw.h5ad')
# gene3c = anndata.read_h5ad('integration/FNIH_DHC_genescore.h5ad')

## Read reference gene expression data

In [81]:
expr = anndata.read_h5ad('../../78.FNIH_DPT_IGM_240813//05.R/integration/FNIH_Multiome_RNA_int.obj.h5ad')
expr

AnnData object with n_obs × n_vars = 408178 × 3000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'nCount_RNA_raw', 'nFeature_RNA_raw', 'donor_demux', 'nCount_SCT', 'nFeature_SCT', 'SCT.weight', 'seurat_clusters', 'log_nCount_SCT', 'log_nFeature_SCT', 'lane', 'batch', 'gex_raw_reads', 'gex_mapped_reads', 'gex_conf_intergenic_reads', 'gex_conf_exonic_reads', 'gex_conf_intronic_reads', 'gex_conf_exonic_unique_reads', 'gex_conf_exonic_antisense_reads', 'gex_conf_exonic_dup_reads', 'gex_exonic_umis', 'gex_conf_intronic_unique_reads', 'gex_conf_intronic_antisense_reads', 'gex_conf_intronic_dup_reads', 'gex_intronic_umis', 'gex_conf_txomic_unique_reads', 'gex_umis_count', 'gex_genes_count', 'atac_raw_reads', 'atac_unmapped_reads', 'atac_lowmapq', 'atac_dup_reads', 'atac_chimeric_reads', 'atac_mitochondrial_reads', 'atac_fragments', 'atac_TSS_fragments', 'atac_peak_region_fragments', 'atac_peak_region_cutsites', 'TSS.enrichment', 'TSS.percentile', 'condition', 'disease_sta

In [50]:
## Read raw count object
expr = anndata.read_h5ad('../../78.FNIH_DPT_IGM_240813//05.R/integration/FNIH_Multiome_RNA_int.obj.h5ad')
pca_loadings = pd.read_csv("../../78.FNIH_DPT_IGM_240813/05.R/integration/coembed/FNIH_Multiome_RNA_pca_loadings.csv", index_col=0)
pca_embedding = pd.read_csv("../../78.FNIH_DPT_IGM_240813/05.R/integration/coembed/FNIH_Multiome_RNA_pca_embeddings.csv", index_col=0)
stdev = pd.read_table("../../78.FNIH_DPT_IGM_240813/05.R/integration/coembed/FNIH_Multiome_RNA_pca_stdev.csv", index_col=0, names = ['stdev'])

genefilter = np.intersect1d(np.array(pca_loadings.index), gene3c.var_names)
gene3c = gene3c[:, genefilter]

from scipy.sparse import csr_matrix
import scanpy as sc
sc.pp.regress_out(gene3c, ['mapped'])
sc.pp.scale(gene3c, max_value=10)

pca_loadings = pca_loadings.loc[gene3c.var_names]
pca_transformed = np.dot(gene3c.X, pca_loadings.values)
X_weighted = pca_transformed/np.array(stdev.index) # weight.by.var = TRUE
gene3c.obsm['X_pca'] = X_weighted
gene3c

/home/y2xie/miniconda3/envs/seurat/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:668: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


AnnData object with n_obs × n_vars = 97093 × 2990
    obs: 'total', 'mapped', 'unmapped', 'duplicate', 'cis', 'cis_1kb-', 'cis_1kb+', 'cis_10kb+', 'trans'
    var: 'mean', 'std'
    obsm: 'X_pca'

In [51]:
expr = expr[:,gene3c.var_names]
expr.obsm['X_pca'] = pca_embedding.values

/tmp/ipykernel_1927003/2945877213.py:2: ImplicitModificationWarning: Setting element `.obsm['X_pca']` of view, initializing view as actual.
  expr.obsm['X_pca'] = pca_embedding.values


In [52]:
### run integration
from ALLCools.integration.seurat_class import SeuratIntegration

expr.obs['Modality'] = 'RNA'
gene3c.obs['Modality'] = '3C'
adata_list = [expr, gene3c]
integrator = SeuratIntegration()
anchor = integrator.find_anchor(adata_list,
                                k_local=None,
                                key_local='X_pca',
                                k_anchor=15,
                                key_anchor='X',
                                dim_red='cca',
                                max_cc_cells=50000,
                                k_score=15,
                                k_filter=None,
                                scale1=False,
                                scale2=False,
                                n_components=30,
                                n_features=200,
                                alignments=[[[0], [1]]])

Find anchors across datasets.
Run CCA
non zero dims 30
Find Anchors using k=15
Score Anchors
Identified 300600 anchors between datasets 0 and 1.


In [53]:
corrected = integrator.integrate(key_correct='X_pca',
                                 row_normalize=True,
                                 n_components=30,
                                 k_weight=15,
                                 sd=1,
                                 alignments=[[[0], [1]]])
ncell = np.sum([xx.shape[0] for xx in adata_list])
adata_merge2 = anndata.AnnData(
    X=np.ones((ncell, 1)), obs=pd.concat([xx.obs for xx in adata_list], axis=0)
)
adata_merge2.obsm['cca'] = np.concatenate(corrected, axis=0)
adata_merge2.obsm['X_pca'] = normalize(adata_merge2.obsm['cca'][:, :30], axis=1)

adata_merge2

Merge datasets
[[0], [1]]
Initialize
Find nearest anchors. k_weight:  15


/home/y2xie/miniconda3/envs/seurat/lib/python3.11/site-packages/scipy/sparse/_index.py:143: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)


Normalize graph
Transform data


/home/y2xie/miniconda3/envs/seurat/lib/python3.11/site-packages/ALLCools/integration/seurat_class.py:657: RuntimeWarning: invalid value encountered in divide
  D = (1 - D / D[:, -1][:, None]) * score[G]


AnnData object with n_obs × n_vars = 505271 × 1
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'nCount_RNA_raw', 'nFeature_RNA_raw', 'donor_demux', 'nCount_SCT', 'nFeature_SCT', 'SCT.weight', 'seurat_clusters', 'log_nCount_SCT', 'log_nFeature_SCT', 'lane', 'batch', 'gex_raw_reads', 'gex_mapped_reads', 'gex_conf_intergenic_reads', 'gex_conf_exonic_reads', 'gex_conf_intronic_reads', 'gex_conf_exonic_unique_reads', 'gex_conf_exonic_antisense_reads', 'gex_conf_exonic_dup_reads', 'gex_exonic_umis', 'gex_conf_intronic_unique_reads', 'gex_conf_intronic_antisense_reads', 'gex_conf_intronic_dup_reads', 'gex_intronic_umis', 'gex_conf_txomic_unique_reads', 'gex_umis_count', 'gex_genes_count', 'atac_raw_reads', 'atac_unmapped_reads', 'atac_lowmapq', 'atac_dup_reads', 'atac_chimeric_reads', 'atac_mitochondrial_reads', 'atac_fragments', 'atac_TSS_fragments', 'atac_peak_region_fragments', 'atac_peak_region_cutsites', 'TSS.enrichment', 'TSS.percentile', 'condition', 'disease_status

In [54]:
# pd.DataFrame(adata_merge2.obsm['cca'], index = adata_merge2.obs_names).to_csv('integration/FNIH_DHC_genescore_sct_embedding.csv', sep = '\t')
adata_merge2.write("integration/FNIH_DHC_genescore_joint_embedding.h5ad")

... storing 'orig.ident' as categorical
... storing 'donor_demux' as categorical
... storing 'lane' as categorical
... storing 'batch' as categorical
... storing 'condition' as categorical
... storing 'disease_status' as categorical
... storing 'library' as categorical
... storing 'amulet' as categorical
... storing 'Description' as categorical
... storing 'Gender' as categorical
... storing 'Alcoholic..2..Drinks.Day.' as categorical
... storing 'Steatosis..' as categorical
... storing 'Fat.distribution' as categorical
... storing 'Portal.inflammation' as categorical
... storing 'Hepatocyte.necrosis' as categorical
... storing 'Diagnosis' as categorical
... storing 'barcode' as categorical
... storing 'cellsubtype' as categorical
... storing 'celltype' as categorical
... storing 'Fibrosis.stage' as categorical
... storing 'Modality' as categorical


In [56]:
rna_cell = (adata_merge2.obs['Modality']=='RNA')
hic_cell = (adata_merge2.obs['Modality']=='3C')
print(rna_cell.sum(), hic_cell.sum())

import pynndescent
import time
start_time = time.time()
index = pynndescent.NNDescent(adata_merge2.obsm['cca'][rna_cell], metric='euclidean', 
                              n_neighbors=15, random_state=0, n_jobs=-1)
print(time.time() - start_time)
G, D = index.query(adata_merge2.obsm['cca'][hic_cell], k=15)
print(time.time() - start_time)

408178 97093
39.16130614280701
83.4043390750885


In [57]:
chunk_size = 50000
sd = 1

### follow Jingtian's tutorial
### to prevent duplicated point appear
cellfilter = D[:, -1] == 0
D = 1 - D / D[:, -1][:, None]
D[cellfilter] = 1
D = 1 - np.exp(-D * (sd**2) / 4)
D = D / (np.sum(D, axis=1) + 1e-6)[:, None]

In [58]:
rna_cell = rna_cell.index[rna_cell]
hic_cell = hic_cell.index[hic_cell]

enc = OneHotEncoder()
llabel = 'celltype'
labelref = enc.fit_transform(adata_merge2.obs.loc[rna_cell, llabel].astype(str).to_numpy()[:, None])
cluster = pd.DataFrame(index=hic_cell, columns=['rnatype', 'score'], dtype=str)
nn = []
for chunk_start in range(0, len(hic_cell), chunk_size):
    result = (
        D[chunk_start : (chunk_start + chunk_size), :, None]
        * labelref[G[chunk_start : (chunk_start + chunk_size)].flatten()]
        .toarray()
        .reshape((-1, 15, enc.categories_[0].shape[0]))
    ).sum(axis=1)
    result = pd.DataFrame(
        result,
        columns=enc.categories_[0],
        index=hic_cell[chunk_start : (chunk_start + chunk_size)],
    )
    result = result.loc[:, result.columns != "nan"]
    nn.append(result)
    cluster.loc[result.index, "rnatype"] = result.idxmax(axis=1).values
    cluster.loc[result.index, "score"] = result.max(axis=1).values
    print(chunk_start)

print(time.time() - start_time)

0
50000
84.05134153366089


In [59]:
cluster.to_csv('integration/FNIH_DHC_genescore_prediction.prediction.csv')